# Interactive GPA Prediction Demonstration

This demonstration turns the selected semester GPA-change model into a reader-facing prediction form. It answers a practical comparison question: **how does a prediction change when the same academic context is evaluated with or without the recorded AI-related variables?**

The target is continuous semester GPA change:

`GPA change = post-semester GPA - previous-semester GPA`

Use **Context + AI** for the full selected model, **Context only** to exclude direct AI-use information, or **Compare both** to view both predictions for the same academic context. The illustrative post-semester GPA adds the predicted change to the entered previous GPA; it is not a measured future result.

> **Evidence boundary.** The supplied data have undocumented collection and real-versus-synthetic provenance. Predictions are associations learned from this file, not causal estimates, academic advice, or evidence that changing AI use will change a student's GPA.

## 1. Reproducible setup and dependency check

The demonstration uses the workspace environment and does not install packages while running. If the dependency check fails, activate the workspace environment and install the stated widget version before restarting the kernel.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import HTML, display

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError(
        "This demonstration requires ipywidgets 8.1.8 in the workspace .venv. "
        "Install it with: .venv\\Scripts\\python.exe -m pip install ipywidgets==8.1.8"
    ) from exc

if widgets.__version__ != "8.1.8":
    warnings.warn(
        f"This demonstration was verified with ipywidgets 8.1.8; found {widgets.__version__}."
    )

RANDOM_STATE = 42
TEST_SIZE = 0.20
print(f"Dependency check passed: ipywidgets {widgets.__version__}")

Dependency check passed: ipywidgets 8.1.8


## 2. Leakage-safe data and matched model pipelines

The raw CSV is read without modification. Both pipelines use the same deterministic 80/20 row split and the same tuned histogram gradient boosting configuration. Their only intended difference is the information available at prediction time.

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def find_dataset():
    relative = Path("Datasets/Impact of AI on Students/ai_student_impact_dataset.csv")
    for start in (Path.cwd(), *Path.cwd().parents):
        candidate = start / relative
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "The student-impact dataset could not be located from the current folder or its parents."
    )


DATA_PATH = find_dataset()
df = pd.read_csv(DATA_PATH)
df["GPA_Change"] = df["Post_Semester_GPA"] - df["Pre_Semester_GPA"]

CONTEXT_FEATURES = [
    "Pre_Semester_GPA",
    "Major_Category",
    "Year_of_Study",
    "Traditional_Study_Hours",
    "Anxiety_Level_During_Exams",
]

AI_FEATURES = [
    "Weekly_GenAI_Hours",
    "Primary_Use_Case",
    "Prompt_Engineering_Skill",
    "Tool_Diversity",
    "Paid_Subscription",
    "Perceived_AI_Dependency",
    "Institutional_Policy",
]

FULL_FEATURES = CONTEXT_FEATURES + AI_FEATURES
FORBIDDEN_FEATURES = {
    "Student_ID",
    "Post_Semester_GPA",
    "Skill_Retention_Score",
    "Burnout_Risk_Level",
}

assert not FORBIDDEN_FEATURES.intersection(FULL_FEATURES)
assert len(FULL_FEATURES) == len(set(FULL_FEATURES))

train_index, test_index = train_test_split(
    np.arange(len(df)), test_size=TEST_SIZE, random_state=RANDOM_STATE
)
y_train = df.iloc[train_index]["GPA_Change"]
y_test = df.iloc[test_index]["GPA_Change"]


def make_preprocessor(frame):
    categorical = [
        column
        for column in frame.columns
        if pd.api.types.is_string_dtype(frame[column])
        or pd.api.types.is_bool_dtype(frame[column])
    ]
    numeric = [column for column in frame.columns if column not in categorical]
    return ColumnTransformer(
        [
            (
                "numeric",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric,
            ),
            (
                "categorical",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "onehot",
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        ),
                    ]
                ),
                categorical,
            ),
        ]
    )


def make_pipeline(features):
    frame = df.iloc[train_index][features]
    return Pipeline(
        [
            ("preprocess", make_preprocessor(frame)),
            (
                "model",
                HistGradientBoostingRegressor(
                    max_iter=180,
                    learning_rate=0.05,
                    max_leaf_nodes=15,
                    min_samples_leaf=10,
                    l2_regularization=0.1,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


context_pipeline = make_pipeline(CONTEXT_FEATURES)
combined_pipeline = make_pipeline(FULL_FEATURES)
context_pipeline.fit(df.iloc[train_index][CONTEXT_FEATURES], y_train)
combined_pipeline.fit(df.iloc[train_index][FULL_FEATURES], y_train)

MODEL_REGISTRY = {
    "context": {
        "pipeline": context_pipeline,
        "features": CONTEXT_FEATURES,
        "label": "Context only",
    },
    "combined": {
        "pipeline": combined_pipeline,
        "features": FULL_FEATURES,
        "label": "Context + AI",
    },
}

print(f"Loaded {len(df):,} rows; fitted two matched pipelines on {len(train_index):,} training rows.")

Loaded 50,000 rows; fitted two matched pipelines on 40,000 training rows.


## 3. Compact validation summary

The combined pipeline reproduces the selected model's reserved-test results. The context-only row uses the identical tuned model configuration and split, providing a matched demonstration comparison rather than the earlier broad feature-set ablation.

In [3]:
validation_rows = []
for key in ("context", "combined"):
    spec = MODEL_REGISTRY[key]
    prediction = spec["pipeline"].predict(df.iloc[test_index][spec["features"]])
    validation_rows.append(
        {
            "Prediction information": spec["label"],
            "MAE": mean_absolute_error(y_test, prediction),
            "RMSE": mean_squared_error(y_test, prediction) ** 0.5,
            "R-squared": r2_score(y_test, prediction),
        }
    )

validation_summary = pd.DataFrame(validation_rows).set_index("Prediction information")
display(validation_summary.style.format("{:.4f}"))

combined_metrics = validation_summary.loc["Context + AI"]
assert abs(combined_metrics["MAE"] - 0.1112) < 0.00005
assert abs(combined_metrics["RMSE"] - 0.1414) < 0.00005
assert abs(combined_metrics["R-squared"] - 0.4185) < 0.00005

,MAE,RMSE,R-squared
Prediction information,,,
Context only,0.1237,0.1616,0.2405
Context + AI,0.1112,0.1414,0.4185


## 4. Prediction interface independent of the widgets

The function below is the stable inference boundary. Reader-friendly labels are mapped back to the dataset's exact categories and Boolean values, then placed in the precise feature order required by the selected pipeline.

In [4]:
MODE_LABELS = {
    "context": "Context only",
    "combined": "Context + AI",
    "compare": "Compare both",
}

CATEGORY_OPTIONS = {
    "Major_Category": {
        "Arts": "Arts",
        "Business": "Business",
        "Humanities": "Humanities",
        "Medical": "Medical",
        "STEM": "STEM",
    },
    "Year_of_Study": {
        "Freshman": "Freshman",
        "Sophomore": "Sophomore",
        "Junior": "Junior",
        "Senior": "Senior",
        "Graduate": "Graduate",
    },
    "Primary_Use_Case": {
        "Copywriting or drafting": "Copywriting/Drafting",
        "Debugging or troubleshooting": "Debugging/Troubleshooting",
        "Direct answer generation": "Direct_Answer_Generation",
        "Ideation": "Ideation",
        "Summarising reading": "Summarizing_Reading",
    },
    "Prompt_Engineering_Skill": {
        "Beginner": "Beginner",
        "Intermediate": "Intermediate",
        "Advanced": "Advanced",
    },
    "Institutional_Policy": {
        "AI actively encouraged": "Actively_Encouraged",
        "Allowed with citation": "Allowed_With_Citation",
        "Strict ban": "Strict_Ban",
    },
}

NUMERIC_LIMITS = {
    "Pre_Semester_GPA": (0.0, 4.0),
    "Traditional_Study_Hours": (0.0, 40.0),
    "Anxiety_Level_During_Exams": (1, 10),
    "Weekly_GenAI_Hours": (0.0, 40.0),
    "Tool_Diversity": (1, 5),
    "Perceived_AI_Dependency": (1, 10),
}

BOOLEAN_FIELDS = {"Paid_Subscription"}
KNOWN_FIELDS = set(FULL_FEATURES)


def _normalise_profile(profile, features):
    if FORBIDDEN_FEATURES.intersection(profile):
        names = ", ".join(sorted(FORBIDDEN_FEATURES.intersection(profile)))
        raise ValueError(f"Post-outcome or identifier fields are not accepted: {names}")
    unknown = set(profile) - KNOWN_FIELDS
    if unknown:
        raise ValueError(f"Unknown input field(s): {', '.join(sorted(unknown))}")

    missing = [field for field in features if field not in profile]
    if missing:
        raise ValueError(f"Missing required input(s): {', '.join(missing)}")

    clean = {}
    for field in features:
        value = profile[field]
        if field in NUMERIC_LIMITS:
            if isinstance(value, bool) or not isinstance(value, (int, float, np.integer, np.floating)):
                raise ValueError(f"{field} must be numeric.")
            low, high = NUMERIC_LIMITS[field]
            if not np.isfinite(value) or not low <= float(value) <= high:
                raise ValueError(f"{field} must be between {low:g} and {high:g}.")
            clean[field] = int(value) if field in {
                "Anxiety_Level_During_Exams",
                "Tool_Diversity",
                "Perceived_AI_Dependency",
            } else float(value)
        elif field in BOOLEAN_FIELDS:
            if not isinstance(value, (bool, np.bool_)):
                raise ValueError(f"{field} must be True or False.")
            clean[field] = bool(value)
        else:
            allowed = set(CATEGORY_OPTIONS[field].values())
            if value not in allowed:
                raise ValueError(f"{field} must be one of the supported categories.")
            clean[field] = str(value)
    return clean


def _prediction_payload(previous_gpa, prediction, label):
    post_gpa = float(previous_gpa) + float(prediction)
    warnings_list = []
    if not 0.0 <= post_gpa <= 4.0:
        warnings_list.append(
            "The illustrative post-semester GPA is outside the 0-4 scale. "
            "It is shown without clipping because the model predicts GPA change directly."
        )
    if prediction > 0:
        direction = "increase"
    elif prediction < 0:
        direction = "decrease"
    else:
        direction = "no change"
    return {
        "label": label,
        "prediction": float(prediction),
        "post_gpa": post_gpa,
        "direction": direction,
        "warnings": warnings_list,
    }


def _predict_one(profile, registry_key):
    spec = MODEL_REGISTRY[registry_key]
    clean = _normalise_profile(profile, spec["features"])
    row = pd.DataFrame([clean], columns=spec["features"])
    prediction = spec["pipeline"].predict(row)[0]
    return _prediction_payload(profile["Pre_Semester_GPA"], prediction, spec["label"])


def predict_profile(profile, mode):
    """Return prediction details for context, combined, or comparison mode."""
    if mode not in MODE_LABELS:
        raise ValueError("mode must be 'context', 'combined', or 'compare'.")

    if mode in ("context", "combined"):
        selected = _predict_one(profile, mode)
        return {
            "mode": mode,
            "selected_prediction": selected["prediction"],
            "post_gpa": selected["post_gpa"],
            "comparison_prediction": None,
            "comparison_post_gpa": None,
            "delta": None,
            "display_labels": [selected["label"]],
            "predictions": {mode: selected},
            "warnings": selected["warnings"],
        }

    context_result = _predict_one(profile, "context")
    combined_result = _predict_one(profile, "combined")
    warnings_list = list(dict.fromkeys(context_result["warnings"] + combined_result["warnings"]))
    return {
        "mode": "compare",
        "selected_prediction": combined_result["prediction"],
        "post_gpa": combined_result["post_gpa"],
        "comparison_prediction": context_result["prediction"],
        "comparison_post_gpa": context_result["post_gpa"],
        "delta": combined_result["prediction"] - context_result["prediction"],
        "display_labels": [context_result["label"], combined_result["label"]],
        "predictions": {
            "context": context_result,
            "combined": combined_result,
        },
        "warnings": warnings_list,
    }

## 5. Direct acceptance checks and worked example

These checks exercise the prediction function without rendering the interface. They verify the expected example, information isolation, invalid-input rejection, and the out-of-range warning rule.

In [5]:
WORKED_EXAMPLE = {
    "Pre_Semester_GPA": 3.40,
    "Major_Category": "Business",
    "Year_of_Study": "Junior",
    "Traditional_Study_Hours": 14.0,
    "Anxiety_Level_During_Exams": 3,
    "Weekly_GenAI_Hours": 10.0,
    "Primary_Use_Case": "Ideation",
    "Prompt_Engineering_Skill": "Advanced",
    "Tool_Diversity": 3,
    "Paid_Subscription": True,
    "Perceived_AI_Dependency": 3,
    "Institutional_Policy": "Actively_Encouraged",
}

example_result = predict_profile(WORKED_EXAMPLE, "compare")
assert abs(example_result["predictions"]["context"]["prediction"] - 0.2667565435) < 0.0005
assert abs(example_result["predictions"]["combined"]["prediction"] - 0.3347486366) < 0.0005
assert abs(example_result["delta"] - 0.0679920931) < 0.0005

changed_hidden_ai = dict(WORKED_EXAMPLE)
changed_hidden_ai.update(
    {
        "Weekly_GenAI_Hours": 40.0,
        "Primary_Use_Case": "Direct_Answer_Generation",
        "Prompt_Engineering_Skill": "Beginner",
        "Tool_Diversity": 5,
        "Paid_Subscription": False,
        "Perceived_AI_Dependency": 10,
        "Institutional_Policy": "Strict_Ban",
    }
)
assert predict_profile(WORKED_EXAMPLE, "context")["selected_prediction"] == predict_profile(
    changed_hidden_ai, "context"
)["selected_prediction"]

for invalid_profile in (
    {**WORKED_EXAMPLE, "Pre_Semester_GPA": 4.1},
    {**WORKED_EXAMPLE, "Paid_Subscription": "Yes"},
    {**WORKED_EXAMPLE, "Post_Semester_GPA": 3.7},
):
    try:
        predict_profile(invalid_profile, "combined")
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid programmatic input was not rejected.")

assert _prediction_payload(3.95, 0.10, "Warning test")["warnings"]

display(
    pd.DataFrame(
        [
            {
                "Prediction information": item["label"],
                "Predicted GPA change": item["prediction"],
                "Illustrative post-semester GPA": item["post_gpa"],
            }
            for item in example_result["predictions"].values()
        ]
    ).style.format(
        {
            "Predicted GPA change": "{:+.3f}",
            "Illustrative post-semester GPA": "{:.2f}",
        }
    )
)
print(f"Prediction difference when additional information is included: {example_result['delta']:+.3f}")

,Prediction information,Predicted GPA change,Illustrative post-semester GPA
0,Context only,+0.267,3.67
1,Context + AI,+0.335,3.73


Prediction difference when additional information is included: +0.068


## 6. Interactive demonstration

Choose a prediction mode, adjust the bounded controls, and select **Run prediction**. **Load worked example** restores the report example; **Reset** returns to neutral defaults. In context-only mode, the AI-related card is hidden and its values are ignored by the prediction pipeline.

In [6]:
display(
    HTML(
        """
<style>
.demo-shell {font-family: Inter, Segoe UI, Arial, sans-serif; color:#17243b; max-width:1120px; margin:0 auto;}
.demo-header {background:linear-gradient(135deg,#102a43,#183b56); color:white; border-radius:16px; padding:22px 26px; box-shadow:0 8px 24px rgba(16,42,67,.18);}
.demo-eyebrow {font-size:12px; letter-spacing:.12em; text-transform:uppercase; color:#82e6d2; font-weight:700;}
.demo-title {font-size:25px; font-weight:750; margin:5px 0 6px;}
.demo-subtitle {font-size:14px; line-height:1.5; color:#e7eef5; max-width:850px;}
.mode-card,.input-card,.action-card,.results-panel {background:#f7f9fc; border:1px solid #d8e2ec; border-radius:14px; padding:17px; margin-top:14px; box-shadow:0 3px 12px rgba(16,42,67,.06);}
.mode-card {background:#eef5f8; border-left:5px solid #159a9c;}
.input-grid {gap:14px; align-items:stretch; flex-flow:row wrap !important;}
.input-card {flex:1 1 390px; min-width:320px; margin-top:14px; overflow:hidden !important;}
.context-card {border-top:4px solid #159a9c;}
.ai-card {border-top:4px solid #f0a202;}
.card-heading {font-size:16px; font-weight:750; margin:0 0 3px; color:#102a43;}
.card-note {font-size:12px; color:#5c6f82; line-height:1.4; margin-bottom:10px;}
.widget-label {font-weight:600;}
.action-card {display:flex; align-items:center; gap:10px; background:white;}
.primary-action button {background:#102a43 !important; color:white !important; border:none !important; font-weight:700 !important; min-height:42px;}
.secondary-action button {background:white !important; color:#102a43 !important; border:1px solid #9fb3c8 !important; min-height:40px;}
.status-ok {background:#e7f8f4; color:#0b6b61; border-left:4px solid #159a9c; padding:10px 13px; border-radius:8px; margin-top:10px;}
.status-error {background:#fff1e8; color:#8a3a0a; border-left:4px solid #f0a202; padding:10px 13px; border-radius:8px; margin-top:10px;}
.results-panel {background:#f7f9fc; border-top:5px solid #102a43;}
.result-grid {display:grid; grid-template-columns:repeat(auto-fit,minmax(245px,1fr)); gap:12px; margin:12px 0;}
.result-card {background:white; border:1px solid #d8e2ec; border-radius:12px; padding:15px;}
.result-card.context {border-left:5px solid #159a9c;}
.result-card.combined {border-left:5px solid #f0a202;}
.result-label {font-size:13px; font-weight:700; color:#486581;}
.result-value {font-size:30px; font-weight:800; color:#102a43; margin:5px 0;}
.result-detail {font-size:13px; color:#52677b; line-height:1.5;}
.delta-callout {background:#eef5f8; border-radius:10px; padding:12px 14px; color:#243b53; line-height:1.5;}
.bar-chart {background:white; border:1px solid #d8e2ec; border-radius:10px; padding:13px; margin-top:12px;}
.bar-row {display:grid; grid-template-columns:125px 1fr 62px; gap:9px; align-items:center; margin:8px 0; font-size:12px;}
.bar-track {height:14px; background:#e5ebf1; border-radius:7px; overflow:hidden;}
.bar {height:100%; border-radius:7px; min-width:3px;}
.bar.context {background:#159a9c;}.bar.combined {background:#f0a202;}
.warning-box {background:#fff7e6; color:#7a4b00; border-left:4px solid #f0a202; border-radius:8px; padding:10px 13px; margin-top:10px;}
.evidence-note {font-size:12px; color:#5c6f82; line-height:1.5; margin-top:12px;}
@media (max-width:760px){.input-card{min-width:100%;}.bar-row{grid-template-columns:105px 1fr 55px;}}
</style>
"""
    )
)


def widget_style():
    return {"description_width": "185px"}


def full_width():
    return widgets.Layout(width="100%")


def slider_kwargs(description, tooltip):
    return {
        "description": description,
        "style": widget_style(),
        "layout": full_width(),
        "continuous_update": False,
        "tooltip": tooltip,
    }


mode_control = widgets.ToggleButtons(
    options=[
        ("Context + AI", "combined"),
        ("Context only", "context"),
        ("Compare both", "compare"),
    ],
    value="compare",
    description="Prediction mode",
    style={"description_width": "130px", "button_width": "150px"},
    layout=widgets.Layout(width="100%"),
    tooltips=[
        "Use academic context and recorded AI-related inputs.",
        "Use academic context only; AI inputs are hidden and ignored.",
        "Show both predictions for the same academic context.",
    ],
)

previous_gpa = widgets.FloatSlider(value=3.40, min=0, max=4, step=0.01, readout_format=".2f", **slider_kwargs("Previous GPA", "Previous-semester GPA on a 0-4 scale."))
major = widgets.Dropdown(options=list(CATEGORY_OPTIONS["Major_Category"].items()), value="Business", description="Major", style=widget_style(), layout=full_width(), tooltip="Broad academic major category.")
year = widgets.Dropdown(options=list(CATEGORY_OPTIONS["Year_of_Study"].items()), value="Junior", description="Year of study", style=widget_style(), layout=full_width(), tooltip="Current year or level of study.")
study_hours = widgets.FloatSlider(value=14.0, min=0, max=40, step=0.5, readout_format=".1f", **slider_kwargs("Traditional study (h/week)", "Weekly non-AI study hours."))
anxiety = widgets.IntSlider(value=3, min=1, max=10, step=1, **slider_kwargs("Exam anxiety (1-10)", "Self-reported exam anxiety; 1 is lowest and 10 is highest."))

ai_hours = widgets.FloatSlider(value=10.0, min=0, max=40, step=0.5, readout_format=".1f", **slider_kwargs("GenAI use (h/week)", "Weekly hours using generative AI tools."))
use_case = widgets.Dropdown(options=list(CATEGORY_OPTIONS["Primary_Use_Case"].items()), value="Ideation", description="Primary AI use", style=widget_style(), layout=full_width(), tooltip="The main recorded purpose for using generative AI.")
prompt_skill = widgets.Dropdown(options=list(CATEGORY_OPTIONS["Prompt_Engineering_Skill"].items()), value="Advanced", description="Prompt skill", style=widget_style(), layout=full_width(), tooltip="Self-reported prompt-engineering skill.")
tool_diversity = widgets.IntSlider(value=3, min=1, max=5, step=1, **slider_kwargs("Number of AI tools", "Count of different AI tools used."))
paid_subscription = widgets.Checkbox(value=True, description="Paid AI subscription", indent=False, style={"description_width": "185px"}, layout=full_width(), tooltip="Whether the student has a paid AI subscription.")
dependency = widgets.IntSlider(value=3, min=1, max=10, step=1, **slider_kwargs("Perceived dependency (1-10)", "Self-reported reliance on AI tools; 1 is lowest and 10 is highest."))
policy = widgets.Dropdown(options=list(CATEGORY_OPTIONS["Institutional_Policy"].items()), value="Actively_Encouraged", description="Institutional policy", style=widget_style(), layout=full_width(), tooltip="The recorded institutional AI policy.")


header = widgets.HTML(
    """
<div class="demo-header">
  <div class="demo-eyebrow">Applied machine learning | demonstration</div>
  <div class="demo-title">Interactive GPA change prediction</div>
  <div class="demo-subtitle">Enter one student profile, switch the information available to the model, and compare predictions without changing the academic context.</div>
</div>
"""
)
mode_note = widgets.HTML("<div class='card-heading'>Choose the information available</div><div class='card-note'>The model configuration and training split remain fixed. Only the feature set changes.</div>")
mode_card = widgets.VBox([mode_note, mode_control])
mode_card.add_class("mode-card")

context_card = widgets.VBox(
    [
        widgets.HTML("<div class='card-heading'>Academic context</div><div class='card-note'>Used in every prediction mode.</div>"),
        previous_gpa,
        major,
        year,
        study_hours,
        anxiety,
    ]
)
context_card.add_class("input-card")
context_card.add_class("context-card")

ai_card = widgets.VBox(
    [
        widgets.HTML("<div class='card-heading'>Recorded AI-related information</div><div class='card-note'>Hidden and ignored in context-only mode.</div>"),
        ai_hours,
        use_case,
        prompt_skill,
        tool_diversity,
        paid_subscription,
        dependency,
        policy,
    ]
)
ai_card.add_class("input-card")
ai_card.add_class("ai-card")

input_grid = widgets.HBox([context_card, ai_card])
input_grid.add_class("input-grid")

run_button = widgets.Button(description="Run prediction", icon="play", button_style="", layout=widgets.Layout(width="185px"), tooltip="Evaluate the current profile.")
run_button.add_class("primary-action")
example_button = widgets.Button(description="Load worked example", icon="bookmark", layout=widgets.Layout(width="195px"), tooltip="Load the reproducible report example.")
example_button.add_class("secondary-action")
reset_button = widgets.Button(description="Reset", icon="refresh", layout=widgets.Layout(width="110px"), tooltip="Return all controls to neutral defaults.")
reset_button.add_class("secondary-action")
button_row = widgets.HBox([run_button, example_button, reset_button], layout=widgets.Layout(flex_flow="row wrap", grid_gap="8px"))
button_row.add_class("action-card")

status_widget = widgets.HTML()
result_widget = widgets.HTML()
result_widget.add_class("results-panel")


def current_profile():
    return {
        "Pre_Semester_GPA": previous_gpa.value,
        "Major_Category": major.value,
        "Year_of_Study": year.value,
        "Traditional_Study_Hours": study_hours.value,
        "Anxiety_Level_During_Exams": anxiety.value,
        "Weekly_GenAI_Hours": ai_hours.value,
        "Primary_Use_Case": use_case.value,
        "Prompt_Engineering_Skill": prompt_skill.value,
        "Tool_Diversity": tool_diversity.value,
        "Paid_Subscription": paid_subscription.value,
        "Perceived_AI_Dependency": dependency.value,
        "Institutional_Policy": policy.value,
    }


def result_card(key, payload):
    return f"""
    <div class="result-card {key}">
      <div class="result-label">{payload['label']}</div>
      <div class="result-value">{payload['prediction']:+.3f}</div>
      <div class="result-detail"><b>{payload['direction'].title()}</b> in predicted GPA<br>Illustrative post-semester GPA: <b>{payload['post_gpa']:.2f}</b></div>
    </div>
    """


def render_result(result):
    cards = "".join(result_card(key, payload) for key, payload in result["predictions"].items())
    predictions = list(result["predictions"].items())
    scale = max(0.40, *(abs(item[1]["prediction"]) for item in predictions))
    bars = "".join(
        f"<div class='bar-row'><span>{payload['label']}</span><div class='bar-track'><div class='bar {key}' style='width:{100 * abs(payload['prediction']) / scale:.1f}%'></div></div><b>{payload['prediction']:+.3f}</b></div>"
        for key, payload in predictions
    )
    chart_label = "; ".join(f"{payload['label']} {payload['prediction']:+.3f}" for _, payload in predictions)

    delta_html = ""
    if result["mode"] == "compare":
        delta_html = f"""
        <div class="delta-callout"><b>Prediction difference: {result['delta']:+.3f}</b><br>
        Adding the recorded AI-related information changes this fitted prediction from {result['comparison_prediction']:+.3f} to {result['selected_prediction']:+.3f}. This is an information-based prediction difference, not an estimated AI effect.</div>
        """

    warnings_html = "".join(f"<div class='warning-box'><b>Scale warning:</b> {warning}</div>" for warning in result["warnings"])
    result_widget.value = f"""
    <div class="card-heading">Prediction result</div>
    <div class="card-note">Signed GPA change is shown first; positive values indicate a predicted increase.</div>
    <div class="result-grid">{cards}</div>
    {delta_html}
    <div class="bar-chart" role="img" aria-label="Predicted GPA change comparison: {chart_label}">
      <div class="result-label">Accessible comparison chart | bar length represents absolute predicted change</div>{bars}
    </div>
    {warnings_html}
    <div class="evidence-note"><b>Use with care:</b> These are predictions from the supplied dataset. They are not causal findings, guarantees, or academic advice. The post-semester value is an illustration formed by adding the prediction to the entered previous GPA.</div>
    """


def on_mode_change(change):
    mode = change["new"]
    ai_card.layout.display = "none" if mode == "context" else ""
    result_widget.value = "<div class='card-heading'>Prediction result</div><div class='card-note'>Inputs changed. Select Run prediction to refresh the result.</div>"
    status_widget.value = ""


def run_prediction(_):
    try:
        result = predict_profile(current_profile(), mode_control.value)
        render_result(result)
        status_widget.value = f"<div class='status-ok'><b>Prediction ready.</b> Evaluated in {MODE_LABELS[mode_control.value]} mode.</div>"
    except Exception as exc:
        status_widget.value = f"<div class='status-error'><b>Input error.</b> {exc}</div>"


def load_example(_):
    mode_control.value = "compare"
    previous_gpa.value = WORKED_EXAMPLE["Pre_Semester_GPA"]
    major.value = WORKED_EXAMPLE["Major_Category"]
    year.value = WORKED_EXAMPLE["Year_of_Study"]
    study_hours.value = WORKED_EXAMPLE["Traditional_Study_Hours"]
    anxiety.value = WORKED_EXAMPLE["Anxiety_Level_During_Exams"]
    ai_hours.value = WORKED_EXAMPLE["Weekly_GenAI_Hours"]
    use_case.value = WORKED_EXAMPLE["Primary_Use_Case"]
    prompt_skill.value = WORKED_EXAMPLE["Prompt_Engineering_Skill"]
    tool_diversity.value = WORKED_EXAMPLE["Tool_Diversity"]
    paid_subscription.value = WORKED_EXAMPLE["Paid_Subscription"]
    dependency.value = WORKED_EXAMPLE["Perceived_AI_Dependency"]
    policy.value = WORKED_EXAMPLE["Institutional_Policy"]
    run_prediction(None)


def reset_form(_):
    mode_control.value = "combined"
    previous_gpa.value = 3.00
    major.value = "STEM"
    year.value = "Sophomore"
    study_hours.value = 12.0
    anxiety.value = 5
    ai_hours.value = 5.0
    use_case.value = "Summarizing_Reading"
    prompt_skill.value = "Intermediate"
    tool_diversity.value = 2
    paid_subscription.value = False
    dependency.value = 5
    policy.value = "Allowed_With_Citation"
    result_widget.value = "<div class='card-heading'>Prediction result</div><div class='card-note'>Select Run prediction when the profile is ready.</div>"
    status_widget.value = "<div class='status-ok'><b>Reset complete.</b> Neutral example values are ready.</div>"


mode_control.observe(on_mode_change, names="value")
run_button.on_click(run_prediction)
example_button.on_click(load_example)
reset_button.on_click(reset_form)

demo = widgets.VBox([header, mode_card, input_grid, button_row, status_widget, result_widget])
demo.add_class("demo-shell")
display(demo)

# Save the report example and its result as the executed default state.
load_example(None)

## 7. Reading the demonstration responsibly

- **Context only** intentionally ignores every AI-related value, even if a profile dictionary contains those keys.
- **Context + AI** uses all leakage-safe predictors selected for the reported model.
- **Compare both** holds the academic context fixed and reports the fitted prediction difference after additional recorded information is supplied. It must not be interpreted as the effect of AI use.
- Predictions are most defensible for profiles resembling the supplied data. The interface bounds inputs, but those bounds do not remove uncertainty or provenance limitations.
- The tool is an explanatory model showcase. It is not suitable for grading, student intervention, institutional policy, or individual academic advice.